# IMP Demo — WRDS leverage variant (`demo_wrds_lev`)

Fork of `notebook/demo.ipynb` that keeps the same pipeline sandbox, then in **§3b** pulls Compustat quarterly fundamentals via WRDS (`gvkey` from `data/stock_universe.csv`) to attach `leverage` and `lev_x_tnx` onto `df_align`.

## Prerequisites

- Same data layout as the main demo (`config/config.py` / `data/`).
- Run `python script/export_universe.py` with `USE_WRDS = True` so `stock_universe.csv` has `gvkey`.
- WRDS login available (`WRDS_USERNAME` in §0a, or the local WRDS credential store).
- Run cells **top to bottom**. After changing CONFIG or §3 / §5+, re-run from that section through §7.

## Where to edit

| Goal | Edit here | Notes |
| ---- | --------- | ----- |
| Target / split / benchmark / families | **§0a** | Same as `demo.ipynb`. |
| WRDS account | **§0a** `WRDS_USERNAME` | `None` prompts / uses stored creds. |
| Leverage feature | **§3b** | Uses `gvkey` → `comp.fundq` → `merge_asof` on `df_align`. |
| Model inputs | **§0a** `FEATURE_COLUMNS` | Defaults to `enabled_features()`; uncomment `leverage` / `lev_x_tnx` after §3b. |
| Custom model | **§5+** | Same as main demo. |



## 0. Config

Shared knobs (including which families to run), then hyperparameter cells for
`LIN` / `TR` / `NN`. Those family cells always define settings; only `FAMILIES`
in **0a** decides what actually runs.


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import src.train as _train
importlib.reload(_train)

from src.report import print_notebook_config
from src.train import resolve_demo_setup


In [ ]:
pd.set_option("display.width", None) 
pd.set_option("display.expand_frame_repr", False)  
pd.set_option("display.max_columns", None)

### 0a. Shared


In [ ]:
# Shared across families (target / split / benchmark / importance).
# Notebook-local knobs — pass these explicitly below; do not rely on config.config.
TARGET = "ret_ann_mkt"
# WRDS login for §3b (None → prompt / local credential store).
WRDS_USERNAME = None
# Benchmark for *_mkt labels: "yahoo" | "equal" | "cap"
BENCHMARK_SOURCE = "equal"
# Used only when BENCHMARK_SOURCE == "yahoo".
MARKET_TICKER = "SPY"
# Catalog: enabled_features() plus optional WRDS leverage columns after §3b.
# Comment out a name in config/config.py groups to disable globally;
# uncomment leverage / lev_x_tnx here after §3b runs.
from src.features import enabled_features

FEATURE_COLUMNS = enabled_features() + [
    # "leverage",
    # "lev_x_tnx",
]

SPLIT_COLUMN = "DecisionDate"
TRAIN_END = "2022-12-31"
VAL_END = "2023-12-31"
IMPUTE_STRATEGY = "median"
RANDOM_STATE = 42

# Which families to run (same idea as main.py FAMILIES).
# "custom" is demo-only; define CUSTOM + fit_custom in §5+.
FAMILIES = (
    "lin",
    # "tr",
    # "nn",
    # "custom",
)



### 0b. Linear

Hyperparameters for the linear family (used when `"lin"` is in `FAMILIES`).
Defaults match `config/config_lin.py`.


In [ ]:
LIN = {
    "MODEL_TYPES": ("ols", "ridge", "lasso"),
    "ALPHA": 1.0,
    "ALPHA_GRID": (0.01, 0.1, 1.0, 10.0, 100.0),
    "STANDARDIZE": True,
}


### 0c. Trees

Hyperparameters for the tree family (used when `"tr"` is in `FAMILIES`).
Defaults match `config/config_tr.py`.


In [ ]:
TR = {
    "MODEL_TYPES": (
        "decision_tree",
        "random_forest",
        "hist_gradient_boosting",
    ),
    "N_ESTIMATORS": 200,
    "MAX_DEPTH": 4,
    "MAX_DEPTH_GRID": (2, 3, 4, 6, 8),
    "LEARNING_RATE": 0.05,
    "MIN_SAMPLES_LEAF": 20,
    "EARLY_STOPPING_PATIENCE": 10,
    "STANDARDIZE": False,
}


### 0d. Neural

Hyperparameters for the neural family (used when `"nn"` is in `FAMILIES`).
Defaults match `config/config_nn.py`.


In [ ]:
NN = {
    "MODEL_TYPES": ("mlp_shallow", "mlp", "mlp_deep"),
    "HIDDEN_SIZES": {
        "mlp_shallow": (64,),
        "mlp": (64, 32),
        "mlp_deep": (128, 64, 32),
    },
    "ALPHA": 1e-4,
    "ALPHA_GRID": (1e-5, 1e-4, 1e-3, 1e-2),
    "LR": 1e-3,
    "BATCH_SIZE": 64,
    "MAX_ITER": 200,
    "EARLY_STOPPING_PATIENCE": 8,
    "STANDARDIZE": True,
}


### Resolve Active Families And Data Paths


In [ ]:
setup = resolve_demo_setup(FAMILIES, lin=LIN, tr=TR, nn=NN, root=ROOT)
ACTIVE_FAMILIES = list(setup.active_families)
STANDARDIZE = setup.standardize
DATA_DIR = setup.data_dir


In [ ]:
print_notebook_config(
    target=TARGET,
    feature_columns=FEATURE_COLUMNS,
    split_column=SPLIT_COLUMN,
    train_end=TRAIN_END,
    val_end=VAL_END,
    impute_strategy=IMPUTE_STRATEGY,
    random_state=RANDOM_STATE,
    standardize=STANDARDIZE,
    active_families=ACTIVE_FAMILIES,
    benchmark_source=BENCHMARK_SOURCE,
    market_ticker=MARKET_TICKER,
)

## 1. Load And Merge

Load the three Parquet tables and build the daily panel `df_all` via `src.prepare` (`load_stock`, `load_earnings`, `load_sentiment`, `merge_panel`).

In [ ]:
from src.prepare import load_earnings, load_sentiment, load_stock, merge_panel

In [ ]:
stock_data = load_stock(DATA_DIR)
stock_data.head()

In [ ]:
earnings = load_earnings(DATA_DIR)
earnings.head()

In [ ]:
sentiment = load_sentiment(DATA_DIR)
sentiment.head()

In [ ]:
df_all = merge_panel(stock_data, earnings, sentiment)
df_all

In [ ]:
df_summary = df_all.describe(include="all").T
df_summary["dtype"] = df_all.dtypes
df_summary

## 2. Aligned Panel And Benchmark

Build `df_align` with `src.events.build_aligned_panel` (uses `src.sessions` + `src.benchmark`). Pass §0a `BENCHMARK_SOURCE` / `MARKET_TICKER` explicitly so the demo does not inherit `config.config`.


In [ ]:
from src.events import build_aligned_panel, build_event_table
from src.report import print_event_report


In [ ]:
df_align = build_aligned_panel(
    df_all,
    data_dir=DATA_DIR,
    market_ticker=MARKET_TICKER,
    benchmark_source=BENCHMARK_SOURCE,
)
df_align.head()

## 3. New Customized Data / Features

Yahoo macros (`vix` / `tnx` / `dxy`) are already attached in §2 via `config.config.MACRO_FEATURES` (cached like the yahoo benchmark under `data/macro_daily.*`). To use them in models, include them in `MACRO_FEATURES` (or §0a `FEATURE_COLUMNS`).

Optional: merge any *additional* daily table onto `df_align` by `Date` below.


In [ ]:
# Macros are on df_align already (from MACRO_FEATURES). Preview:
df_align[["Date", "vix", "tnx", "dxy"]].drop_duplicates("Date").head()

In [ ]:
# Example: merge an extra local table (not required for vix/tnx/dxy).
""" extra = pd.read_parquet(DATA_DIR / "my_extra_daily.parquet")
df_align = df_align.merge(extra, on="Date", how="left")
df_align.head() """

Build decision-day features on `df_align`: `mom_21d`, `vol_21d`, `log_mcap` (no lookahead; known at that close).

In [ ]:
""" WINDOW = 21

df_align = df_align.sort_values(["TickerId", "Date"]).reset_index(drop=True)
price = df_align.groupby("TickerId", sort=False)["LastPrice"]
ret_1d = price.pct_change()
mcap = pd.to_numeric(df_align["MarketCap"], errors="coerce")

# Calculate the percentage change over the last 21 days 
df_align["mom_21d"] = price.pct_change(WINDOW)

# Calculate the standard deviation of the daily percentage changes over the last 21 days
df_align["vol_21d"] = (
    ret_1d.groupby(df_align["TickerId"], sort=False)
    .rolling(WINDOW, min_periods=WINDOW)
    .std(ddof=1)
    .reset_index(level=0, drop=True)
)

# Calculate the log of market cap
df_align["log_mcap"] = np.where(mcap > 0, np.log(mcap), np.nan)

df_align.head() """

### 3b. WRDS leverage (`gvkey` → Compustat fundq)

Requires `data/stock_universe.csv` with non-null `gvkey` (from `script/export_universe.py` with `USE_WRDS = True`). Builds quarterly leverage, as-of merges onto `df_align` (report available strictly before each `Date`), then forms a train-centered `leverage × tnx` interaction.

Uncomment `leverage` / `lev_x_tnx` in §0a `FEATURE_COLUMNS` (appended after `enabled_features()`) after this section runs.


In [ ]:
stock_universe = pd.read_csv(DATA_DIR / "stock_universe.csv")
stock_universe = stock_universe.dropna(subset=["gvkey"]).copy()
stock_universe["gvkey"] = (
    stock_universe["gvkey"].astype("int").astype("str").str.zfill(6)
)
stock_universe.head()


In [ ]:
# Rows without gvkey in the raw CSV (for visibility).
pd.read_csv(DATA_DIR / "stock_universe.csv").loc[
    lambda d: d["gvkey"].isna(), ["TickerId", "Symbol", "Exchange", "InstrumentName"]
].head()


In [ ]:
gvkeys = stock_universe["gvkey"].unique().tolist()
gvkeys[:5]


In [ ]:
import wrds

db = wrds.Connection(wrds_username=WRDS_USERNAME) if WRDS_USERNAME else wrds.Connection()


In [ ]:
# Query Quarterly Fundamentals (comp.fundq) using GVKEYs
query = """
SELECT gvkey, datadate, rdq, atq, dlcq, dlttq
FROM comp.fundq
WHERE gvkey = ANY(%s)
  AND datadate >= '2020-01-01'
  AND indfmt = 'INDL' AND datafmt = 'STD' AND popsrc = 'D' AND consol = 'C'
"""

fundq = db.raw_sql(query, params=(gvkeys,), date_cols=["datadate", "rdq"])

debt = fundq[["dlcq", "dlttq"]].sum(axis=1, min_count=1)
fundq["leverage"] = debt / fundq["atq"].where(fundq["atq"] > 0)
fundq["avail_date"] = fundq["rdq"].fillna(fundq["datadate"] + pd.Timedelta(days=90))

lev = (
    fundq.merge(stock_universe[["TickerId", "gvkey"]], on="gvkey")
    .dropna(subset=["leverage", "avail_date"])
    [["TickerId", "avail_date", "leverage"]]
)
lev.head()


In [ ]:
db.close()


In [ ]:
df_align = df_align.drop(columns=["leverage", "lev_x_tnx"], errors="ignore")
df_align["Date"] = pd.to_datetime(df_align["Date"])
lev = lev.copy()
lev["avail_date"] = pd.to_datetime(lev["avail_date"])
lev["TickerId"] = lev["TickerId"].astype(df_align["TickerId"].dtype)

df_align = pd.merge_asof(
    df_align.sort_values("Date"),
    lev.sort_values("avail_date"),
    left_on="Date",
    right_on="avail_date",
    by="TickerId",
    direction="backward",
    allow_exact_matches=False,  # report on the decision day itself does not count
).drop(columns="avail_date")

# Interaction with macro tnx already on df_align from MACRO_FEATURES; center on train.
is_train = df_align["Date"] <= pd.to_datetime(TRAIN_END)
lev_c = df_align["leverage"] - df_align.loc[is_train, "leverage"].mean()
tnx_c = df_align["tnx"] - df_align.loc[is_train, "tnx"].mean()
df_align["lev_x_tnx"] = lev_c * tnx_c

df_align = df_align.sort_values(["TickerId", "Date"]).reset_index(drop=True)
df_align[["TickerId", "Date", "leverage", "tnx", "lev_x_tnx"]].dropna().head()


## 4. Event Table

Build one-row-per-announcement `df` with `src.events.build_event_table` (reuses `df_align`); print descriptive stats with `src.report.print_event_report`.


In [ ]:
df = build_event_table(
    df_all,
    data_dir=DATA_DIR,
    df_align=df_align,
    feature_columns=FEATURE_COLUMNS,
    market_ticker=MARKET_TICKER,
    benchmark_source=BENCHMARK_SOURCE,
)
print_event_report(df, primary_target=TARGET, feature_columns=FEATURE_COLUMNS)


## 5. Preprocess (Train / Val / Test)

Chronological split and train-only impute/scale via `src.preprocess.run_preprocess`; report with `src.report.print_preprocess_report`.


In [ ]:
from src.preprocess import run_preprocess
from src.report import print_preprocess_report

In [ ]:
result = run_preprocess(
    df,
    target=TARGET,
    feature_columns=FEATURE_COLUMNS,
    split_column=SPLIT_COLUMN,
    train_end=TRAIN_END,
    val_end=VAL_END,
    impute_strategy=IMPUTE_STRATEGY,
    standardize=STANDARDIZE,
)
print_preprocess_report(result, target=TARGET, split_column=SPLIT_COLUMN)
train_y_mean = float(result.folds["train"].y.to_numpy(dtype=float).mean())


## 5+. Custom Model (demo-only)

Enable with `"custom"` in `FAMILIES` (§0a). Collaborator blocks map to main-family roles:

| Block | Role (like) | Model id |
| ----- | ----------- | -------- |
| **5+a** | `lin` | `elasticnet` |
| **5+b** | `tr` | `extratrees` |
| **5+c** | `nn` | `mlp_linear` |

Edit only 5+a/b/c. **Wire-up** is infrastructure.


In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import ElasticNet
from sklearn.neural_network import MLPRegressor

from src.preprocess import PreprocessResult
from src.train import FitBundle

# Which custom models to run (comment a line to skip that lin/tr/nn slot).
CUSTOM_MODEL_TYPES = (
    # "elasticnet",   
    # "extratrees",   
    # "mlp_linear",   
)
CUSTOM_STANDARDIZE = True

# Filled by 5+a / 5+b / 5+c; do not edit by hand.
_CUSTOM_FITTERS = {}
_CUSTOM_PARAMS = {}


### 5+a. `lin` slot → `elasticnet`

Custom stand-in for the **linear** family. Edit only this block.


In [ ]:
""" # --- collaborator edit: lin / elasticnet ---
ELASTICNET = {
    "ALPHA": 1.0,
    "L1_RATIO": 0.5,
}


def fit_elasticnet(result: PreprocessResult) -> FitBundle:
    train = result.folds["train"]
    alpha = float(ELASTICNET["ALPHA"])
    l1_ratio = float(ELASTICNET["L1_RATIO"])
    model = ElasticNet(
        alpha=alpha,
        l1_ratio=l1_ratio,
        max_iter=10_000,
        random_state=RANDOM_STATE,
    )
    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))
    return FitBundle(
        model=model,
        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),
        summary_extras={"alpha": alpha, "l1_ratio": l1_ratio},
    )


_CUSTOM_FITTERS["elasticnet"] = fit_elasticnet
_CUSTOM_PARAMS.update(ELASTICNET) """


### 5+b. `tr` slot → `extratrees`

Custom stand-in for the **tree** family. Edit only this block.


In [ ]:
""" # --- collaborator edit: tr / extratrees ---
EXTRATREES = {
    "N_ESTIMATORS": 200,
    "MAX_DEPTH": 6,
    "MIN_SAMPLES_LEAF": 20,
}


def fit_extratrees(result: PreprocessResult) -> FitBundle:
    train = result.folds["train"]
    model = ExtraTreesRegressor(
        n_estimators=int(EXTRATREES["N_ESTIMATORS"]),
        max_depth=int(EXTRATREES["MAX_DEPTH"]),
        min_samples_leaf=int(EXTRATREES["MIN_SAMPLES_LEAF"]),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))
    return FitBundle(
        model=model,
        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),
        summary_extras={
            "n_estimators": int(EXTRATREES["N_ESTIMATORS"]),
            "max_depth": int(EXTRATREES["MAX_DEPTH"]),
        },
    )


_CUSTOM_FITTERS["extratrees"] = fit_extratrees
_CUSTOM_PARAMS.update(EXTRATREES) """


### 5+c. `nn` slot → `mlp_linear`

Custom stand-in for the **neural** family. Edit only this block.
(`activation="identity"` ≈ linear map through a hidden layer.)


In [ ]:
""" # --- collaborator edit: nn / mlp_linear ---
MLP_LINEAR = {
    "MLP_HIDDEN": (32,),
    "MLP_MAX_ITER": 300,
}


def fit_mlp_linear(result: PreprocessResult) -> FitBundle:
    train = result.folds["train"]
    hidden = tuple(MLP_LINEAR["MLP_HIDDEN"])
    model = MLPRegressor(
        hidden_layer_sizes=hidden,
        activation="identity",
        solver="adam",
        max_iter=int(MLP_LINEAR["MLP_MAX_ITER"]),
        random_state=RANDOM_STATE,
    )
    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))
    return FitBundle(
        model=model,
        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),
        summary_extras={"hidden": str(hidden)},
    )


_CUSTOM_FITTERS["mlp_linear"] = fit_mlp_linear
_CUSTOM_PARAMS.update(MLP_LINEAR) """


#### Wire-up

Not a lin/tr/nn model block. Errors if `CUSTOM_MODEL_TYPES` names a model that
was not registered by 5+a/b/c; then builds `fit_custom` / `CUSTOM`.


In [ ]:
_unknown = [name for name in CUSTOM_MODEL_TYPES if name not in _CUSTOM_FITTERS]
if _unknown:
    raise ValueError(
        "CUSTOM_MODEL_TYPES names are not registered in §5+a/b/c "
        f"(no fit_* block): {_unknown}. "
        f"Registered: {sorted(_CUSTOM_FITTERS)}."
    )
_unused = sorted(set(_CUSTOM_FITTERS) - set(CUSTOM_MODEL_TYPES))
if _unused:
    print("note: registered but not in CUSTOM_MODEL_TYPES:", _unused)


def fit_custom(model_type: str, result: PreprocessResult) -> FitBundle:
    try:
        fitter = _CUSTOM_FITTERS[model_type]
    except KeyError as exc:
        raise ValueError(
            f"Unknown custom model_type={model_type!r}; "
            f"registered={sorted(_CUSTOM_FITTERS)}."
        ) from exc
    return fitter(result)


CUSTOM = {
    "MODEL_TYPES": tuple(CUSTOM_MODEL_TYPES),
    "STANDARDIZE": bool(CUSTOM_STANDARDIZE),
    **_CUSTOM_PARAMS,
}

ACTIVE_FAMILIES = [
    ("custom", dict(CUSTOM)) if name == "custom" else (name, block)
    for name, block in ACTIVE_FAMILIES
]
if ACTIVE_FAMILIES and ACTIVE_FAMILIES[0][0] == "custom":
    STANDARDIZE = bool(CUSTOM["STANDARDIZE"])

print("fit_custom ready:", CUSTOM["MODEL_TYPES"])
print("registered:", sorted(_CUSTOM_FITTERS))


## 6. Fit And Predict

For each active family: `lin`/`tr`/`nn` via `apply_demo_block`; `custom` via
`build_demo_custom_spec` + §5+ `fit_custom`. Then re-preprocess and `run_one_model`.


In [ ]:
from src.metrics import infer_task
from src.pipeline import run_one_model
from src.preprocess import run_preprocess
from src.train import apply_demo_block, build_demo_custom_spec

In [ ]:
task = infer_task(TARGET)
if task != "regression":
    raise NotImplementedError(f"Demo supports regression only; got TARGET={TARGET!r}")

summary_rows = []
fit_rows = []
for family, block in ACTIVE_FAMILIES:
    if family == "custom":
        spec = build_demo_custom_spec(
            block,
            fit_one=fit_custom,
            random_state=RANDOM_STATE,
        )
    else:
        spec = apply_demo_block(
            family,
            block,
            random_state=RANDOM_STATE,
        )
    result_f = run_preprocess(
        df,
        target=TARGET,
        feature_columns=FEATURE_COLUMNS,
        split_column=SPLIT_COLUMN,
        train_end=TRAIN_END,
        val_end=VAL_END,
        impute_strategy=IMPUTE_STRATEGY,
        standardize=bool(block["STANDARDIZE"]),
    )
    train_y_mean_f = float(result_f.folds["train"].y.to_numpy(dtype=float).mean())
    print(f"\n### family={family}  standardize={block['STANDARDIZE']}  "
          f"models={list(spec.model_types)}")
    for model_type in spec.model_types:
        row, _predictions, record = run_one_model(
            spec,
            model_type=model_type,
            result=result_f,
            train_y_mean=train_y_mean_f,
            task=task,
        )
        summary_rows.append(row)
        fit_rows.append({"family": family, **record})

summary = pd.DataFrame(summary_rows)
fits = pd.DataFrame(fit_rows)

## 7. Summary

Compact cross-model comparison on the test fold (`print_summary_table`).


In [ ]:
print(summary)
# print(fits)